# Assignment 1: Implement relational operators

Relational algebra tells us **what** a query should do. A database system still needs an algorithm that determines **how** to do it. In this assignment, you will implement four small relational-algebra operations over a supplied in-memory representation.

Complete every `TODO`, run the supplied tests, add the requested tests of your own, and answer the short written questions. Submit the completed notebook through Canvas.

## The representation

Each `Relation` has a `name`, ordered `attributes`, and `tuples`. Each tuple has one value for every attribute. The tuple order makes results deterministic for testing; it is **not** a claim that a relation has an inherent row order. This is a deliberately small teaching representation, not a production DBMS.

In [ ]:
from __future__ import annotations

from collections.abc import Callable
from dataclasses import dataclass
from typing import TypeAlias


Value: TypeAlias = str | int | float | bool | None
Row: TypeAlias = list[Value]
Record: TypeAlias = dict[str, Value]
# Predicates receive named records, not positional tuples.
Predicate: TypeAlias = Callable[[Record], bool]


@dataclass
class Relation:
    """A minimal in-memory representation of a relation."""

    name: str
    attributes: list[str]
    # Relational tuples use Python lists so one-attribute results stay readable.
    tuples: list[Row]

    def __post_init__(self) -> None:
        # Reject malformed relations before an operator tries to use them.
        if not self.name:
            raise ValueError("A relation needs a name.")
        if not self.attributes:
            raise ValueError("A relation needs at least one attribute.")
        if len(set(self.attributes)) != len(self.attributes):
            raise ValueError("Attribute names must be unique.")
        if any(len(row) != len(self.attributes) for row in self.tuples):
            raise ValueError("Each tuple must have one value per attribute.")
        if len({tuple(row) for row in self.tuples}) != len(self.tuples):
            raise ValueError("A relation cannot contain duplicate tuples.")

    def records(self) -> list[Record]:
        """Return tuples as named records so predicates are easy to read."""

        # Pair each positional value with its attribute name.
        return [dict(zip(self.attributes, row)) for row in self.tuples]


def show(relation: Relation) -> None:
    """Print a compact view of a relation for use while debugging."""

    # This prints a human-readable view; it does not change the relation.
    print(relation.name, relation.attributes)
    for row in relation.records():
        print(row)


## Sample relations

The relations use the same student and enrollment ideas as the Week 2 lectures. Notice that `enrollment` has repeated `student_id` values: that repetition is meaningful because one student may enroll in more than one section.

In [ ]:
# student stores facts about individual students.
student = Relation(
    name="student",
    attributes=["student_id", "netid", "major", "class_year"],
    tuples=[
        [2001, "aria7", "CS", 2027],
        [2002, "mateo3", "Math", 2026],
        [2003, "priya11", "CS", 2027],
        [2004, "jamal2", "Biology", 2028],
    ],
)

# enrollment stores one student-section association per tuple.
enrollment = Relation(
    name="enrollment",
    attributes=["student_id", "section_id", "enrolled_on"],
    tuples=[
        [2001, 501, "2026-08-17"],
        [2002, 501, "2026-08-17"],
        [2001, 610, "2026-08-18"],
    ],
)

show(student)


## Task 1: selection

Selection retains the tuples that satisfy a predicate. Implement `select`.

- Call `predicate` once for each tuple, using a named record such as `{'student_id': 2001, 'major': 'CS', ...}`.
- Return a **new** `Relation` with the same name and attributes as the input.
- Do not modify the input relation.

In [ ]:
def select(relation: Relation, predicate: Predicate) -> Relation:
    """Return tuples from relation for which predicate returns True."""

    # TODO: Use relation.records(), retain matching tuple lists,
    # and return a new Relation without modifying the input relation.
    raise NotImplementedError


In [ ]:
# Run after implementing select. This also checks that student was not changed.
cs_students = select(student, lambda row: row["major"] == "CS")

assert cs_students == Relation(
    name="student",
    attributes=["student_id", "netid", "major", "class_year"],
    tuples=[[2001, "aria7", "CS", 2027], [2003, "priya11", "CS", 2027]],
)
assert student.tuples == [
    [2001, "aria7", "CS", 2027],
    [2002, "mateo3", "Math", 2026],
    [2003, "priya11", "CS", 2027],
    [2004, "jamal2", "Biology", 2028],
]


## Task 2: projection

Projection retains selected attributes. It can turn distinct input tuples into identical output tuples, so relational projection removes duplicate output tuples.

Implement `project`.

- `attributes` gives the output attribute order.
- Raise `KeyError` if a requested attribute does not exist in the input relation.
- Raise `ValueError` if an attribute is requested more than once.
- Return a new relation with duplicate output tuples removed. Keep the first occurrence so the tests have deterministic output.

In [ ]:
def project(relation: Relation, attributes: list[str]) -> Relation:
    """Return relation with only the requested attributes."""

    # TODO: Find each requested attribute's position, build output lists,
    # remove duplicates, and return a new Relation.
    raise NotImplementedError


In [ ]:
# Run after implementing project. The first assertion checks duplicate removal.
enrolled_students = project(enrollment, ["student_id"])

assert enrolled_students == Relation(
    name="enrollment",
    attributes=["student_id"],
    tuples=[[2001], [2002]],
)
assert list(project(student, ["netid", "student_id"]).records()[0]) == [
    "netid",
    "student_id",
]


## Task 3: rename

Relational algebra can rename a relation without changing its tuples. Rename is especially useful when a larger expression needs two distinct names for the same source relation.

Implement `rename`. Return a new relation named `new_name`, preserve its attributes and tuples exactly, and do not modify the input relation.

In [ ]:
def rename(relation: Relation, new_name: str) -> Relation:
    """Return relation with a new relation name."""

    # TODO: Construct a new Relation with only its name changed.
    raise NotImplementedError


In [ ]:
# Run after implementing rename. The assertions check that no tuple changed.
s = rename(student, "s")

assert s.name == "s"
assert s.attributes == student.attributes
assert s.tuples == student.tuples
assert student.name == "student"


## Task 4: join

An inner equijoin combines every pair of tuples whose specified attribute values match. Implement `join` so that it joins one attribute from `left` with one attribute from `right`.

- Raise `KeyError` if either named join attribute is absent.
- The output attributes are all `left` attributes followed by all `right` attributes except `right_attribute`; this avoids storing the matched value twice.
- Name the result `left.name + '_join_' + right.name`.
- Return a new relation containing every matching pair. Use left-relation order, then right-relation order, so the tests are deterministic.

In [ ]:
def join(
    left: Relation,
    right: Relation,
    left_attribute: str,
    right_attribute: str,
) -> Relation:
    """Return matching tuple pairs from left and right."""

    # TODO: Find the two join positions, compare every left/right pair,
    # and construct each matching output list without right_attribute.
    raise NotImplementedError


In [ ]:
# Run after implementing join. Student 2001 matches two enrollment tuples.
student_enrollment = join(student, enrollment, "student_id", "student_id")

assert student_enrollment == Relation(
    name="student_join_enrollment",
    attributes=["student_id", "netid", "major", "class_year", "section_id", "enrolled_on"],
    tuples=[
        [2001, "aria7", "CS", 2027, 501, "2026-08-17"],
        [2001, "aria7", "CS", 2027, 610, "2026-08-18"],
        [2002, "mateo3", "Math", 2026, 501, "2026-08-17"],
    ],
)


## Task 5: compose operations

The expression below means: "return the netids of CS students." Complete the code by composing your functions. The inner operation should be `select`, and the outer operation should be `project`.

In [ ]:
# TODO: Apply select first so major is still available, then project netid.
cs_netids = None

assert cs_netids == Relation(
    name="student",
    attributes=["netid"],
    tuples=[["aria7"], ["priya11"]],
)


## Required tests

Implement the following two tests in the cells below.

1. **Selection test:** verify that selecting students whose `class_year` is `2028` returns the one tuple for Jamal.
2. **Invalid projection test:** verify that `project(student, ["missing_attribute"])` raises `KeyError`. Your test must fail if no exception is raised.

In [ ]:
# Required test 1: select the one student whose class_year is 2028.
# TODO: Write an assertion for the expected Relation.


In [ ]:
# Required test 2: project an unknown attribute and verify that KeyError is raised.
# TODO: Use try/except (or an equivalent approach) so the test fails if no exception is raised.


## Written responses

Answer each prompt in 2-4 sentences below it.

1. In Task 5, what does the inner `select` operation do? What does the outer `project` operation do?

   **Response:**

2. Why does projecting `student_id` from `enrollment` produce two tuples instead of three?

   **Response:**

3. Why does the join result contain two tuples for student 2001 but none for student 2003?

   **Response:**

4. `select` must inspect every input tuple in this representation. What information or data structure could let a DBMS avoid inspecting every tuple for some predicates?

   **Response:**

5. The logical request "return the netids of CS students" does not require one particular Python implementation. Describe one implementation detail that could change without changing the request's meaning.

   **Response:**

## Grading

This assignment is worth **100 points**.

| Component | Points |
|---|---:|
| Task 1: `select` | 15 |
| Task 2: `project` | 20 |
| Task 3: `rename` | 10 |
| Task 4: `join` | 25 |
| Task 5: composed `select` and `project` | 10 |
| Required selection test | 5 |
| Required invalid-projection test | 5 |
| Written responses (2 points each) | 10 |
| **Total** | **100** |